# Resource Limits & Constraints

By default, a container can use as much CPU, memory, and I/O as the host allows. On a shared host, one runaway container can starve every other workload. Resource limits are how you enforce isolation and protect the host.

Docker resource constraints are implemented via Linux **cgroups** (control groups) — the same kernel feature used by Kubernetes, systemd, and every other container runtime.

Topics:
- Memory limits: `--memory`, `--memory-swap`, OOM behaviour
- CPU limits: `--cpus`, `--cpu-shares`, `--cpuset-cpus`
- Block I/O limits
- `--pids-limit` — preventing fork bombs
- Verifying limits from inside a container
- Setting limits after the fact with `docker update`
- Choosing the right limits in practice

## 1. Memory Limits

### `--memory` / `-m`

Sets the maximum amount of RAM the container can use. Accepts `b`, `k`, `m`, `g` suffixes.

```bash
docker run -m 512m myapp     # 512 MB RAM limit
docker run -m 2g   myapp     # 2 GB RAM limit
```

When the container exceeds its memory limit, the Linux OOM (Out Of Memory) killer terminates the process. Docker marks the container as `OOMKilled: true` in its state.

### `--memory-swap`

Controls the total amount of **memory + swap** the container can use:

| `--memory` | `--memory-swap` | Effect |
|------------|-----------------|--------|
| `512m` | not set | 512m RAM + 512m swap (default: 2× RAM) |
| `512m` | `512m` | 512m RAM + **no swap** (swap = memory = 512m) |
| `512m` | `1g` | 512m RAM + 512m swap |
| `512m` | `-1` | 512m RAM + **unlimited swap** |

Setting `--memory-swap` equal to `--memory` disables swap entirely for the container — good for predictable, latency-sensitive workloads.

### `--memory-reservation`

A soft limit — Docker won't enforce it strictly, but when the host is under memory pressure, the container is expected to stay within this amount. Must be lower than `--memory`.

### OOM behaviour

```bash
# Default: OOM kill terminates the container
docker run -m 64m myapp

# Disable OOM kill — container process is not killed, but the host may
# become unstable. Use with extreme care.
docker run -m 64m --oom-kill-disable myapp

# Adjust OOM score — higher = more likely to be killed first
docker run --oom-score-adj 500 myapp
```

In [ ]:
# Start a container with a 64 MB memory limit
!docker run -d --name mem-demo -m 64m nginx:alpine

# Verify the limit is applied — cgroup exposes it inside the container
# cgroup v2 path: /sys/fs/cgroup/memory.max
# cgroup v1 path: /sys/fs/cgroup/memory/memory.limit_in_bytes
!docker exec mem-demo sh -c '
    if [ -f /sys/fs/cgroup/memory.max ]; then
        echo "cgroup v2 limit: $(cat /sys/fs/cgroup/memory.max) bytes"
    else
        echo "cgroup v1 limit: $(cat /sys/fs/cgroup/memory/memory.limit_in_bytes) bytes"
    fi
'

# Also visible via docker inspect
!docker inspect mem-demo --format 'Memory limit: {{.HostConfig.Memory}} bytes'

!docker rm -f mem-demo

In [ ]:
# Demonstrate OOM kill: try to allocate more than the limit
# This Python process will be killed by the OOM killer
!docker run --rm -m 32m --name oom-demo python:3.12-slim \
    python -c "
import sys
print('Allocating memory beyond the limit...')
sys.stdout.flush()
data = bytearray(200 * 1024 * 1024)  # try to allocate 200 MB
print('Should not reach here')
" 2>&1 || echo "Exit code: $? (non-zero = OOM killed or error)"

# Check OOMKilled flag
# (container is removed by --rm so we can't inspect it after, but you'd check:
# docker inspect CONTAINER --format '{{.State.OOMKilled}}')

## 2. CPU Limits

### `--cpus` — hard CPU limit

Limits the container to a fraction of the host's total CPU. `--cpus 1.5` means the container can use at most 1.5 CPU cores worth of compute, regardless of how many cores the host has.

```bash
docker run --cpus 0.5 myapp    # max 50% of one CPU core
docker run --cpus 2   myapp    # max 2 full CPU cores
docker run --cpus 1.5 myapp    # max 1.5 CPU cores
```

Implemented via the cgroup CPU quota: `--cpus 1.5` sets a 150,000 µs quota per 100,000 µs period.

### `--cpu-shares` — relative weight (soft limit)

Controls the **relative** share of CPU time when the host is under contention. Default is 1024.

```bash
# web gets twice as much CPU as worker when the host is contended
docker run --cpu-shares 2048 --name web    myapp
docker run --cpu-shares 1024 --name worker myworker
```

When the host has spare capacity, both containers can use as much as they want — shares only kick in under contention. Use `--cpus` for a hard ceiling, `--cpu-shares` for relative prioritisation.

### `--cpuset-cpus` — pin to specific cores

Restricts the container to run only on specific CPU cores. Useful for NUMA-aware deployments or preventing CPU cache thrashing.

```bash
docker run --cpuset-cpus 0,1   myapp   # cores 0 and 1 only
docker run --cpuset-cpus 0-3   myapp   # cores 0, 1, 2, 3
docker run --cpuset-cpus 4-7   myapp   # cores 4, 5, 6, 7
```

In [ ]:
# Start two containers: one with a CPU limit, one without
# Both run a CPU-intensive workload — sha256sum over /dev/urandom

!docker run -d --name cpu-limited --cpus 0.5 \
    alpine sh -c 'while true; do dd if=/dev/urandom bs=1M count=10 2>/dev/null | sha256sum; done'

!docker run -d --name cpu-unlimited \
    alpine sh -c 'while true; do dd if=/dev/urandom bs=1M count=10 2>/dev/null | sha256sum; done'

import time; time.sleep(2)

# Compare CPU usage — limited container should be near 50%
!docker stats cpu-limited cpu-unlimited --no-stream \
    --format 'table {{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}'

!docker rm -f cpu-limited cpu-unlimited

In [ ]:
# Verify cpuset from inside the container
import subprocess
host_cpus = int(subprocess.check_output(["nproc"]).strip())
print(f"Host has {host_cpus} CPU cores")

if host_cpus >= 2:
    !docker run --rm --cpuset-cpus 0 alpine \
        sh -c 'cat /sys/fs/cgroup/cpuset.cpus 2>/dev/null || cat /sys/fs/cgroup/cpuset/cpuset.cpus'
else:
    print("Single-core host — skipping cpuset demo")

## 3. Block I/O Limits

Docker can throttle disk read and write throughput and IOPS per container.

```bash
# Limit read throughput to 10 MB/s on /dev/sda
docker run --device-read-bps /dev/sda:10mb myapp

# Limit write throughput to 5 MB/s
docker run --device-write-bps /dev/sda:5mb myapp

# Limit read IOPS to 100 per second
docker run --device-read-iops /dev/sda:100 myapp

# Limit write IOPS to 50 per second
docker run --device-write-iops /dev/sda:50 myapp
```

You must specify the actual block device path (e.g. `/dev/sda`, `/dev/nvme0n1`). Find it with:

```bash
df -h /var/lib/docker   # shows which device the Docker data root is on
lsblk
```

**Relative I/O weight** with `--blkio-weight` (10–1000, default 500) — similar to `--cpu-shares`: sets relative priority under contention.

```bash
docker run --blkio-weight 700 myapp-high-priority
docker run --blkio-weight 300 myapp-low-priority
```

## 4. `--pids-limit` — Preventing Fork Bombs

A fork bomb is a process that continuously spawns child processes, eventually exhausting the system's process table and making the host unresponsive.

```bash
# Limit the container to 100 total processes (PIDs)
docker run --pids-limit 100 myapp
```

Without a limit, a misconfigured or malicious container could fork enough processes to crash the host. `--pids-limit` is a cheap, effective safety net.

In [ ]:
# Demonstrate pids-limit: try to spawn more processes than the limit
!docker run --rm --pids-limit 10 --name pids-demo alpine \
    sh -c '
        echo "PID limit in effect"
        # Try to spawn 20 background processes — will fail after the limit
        i=0
        while [ $i -lt 20 ]; do
            sleep 10 &
            i=$((i+1))
        done 2>&1 | head -5
        echo "Processes spawned before limit hit: $(ls /proc | grep -c ^[0-9] 2>/dev/null || echo unknown)"
    ' 2>&1 | head -10

## 5. Combining Limits

In practice, you apply several limits together. Here's a realistic production-style run command:

In [ ]:
production_run = '''\
docker run -d \\
    --name api-prod \\
    \\
    # Memory: 512 MB hard limit, no swap, reservation at 256 MB
    --memory 512m \\
    --memory-swap 512m \\
    --memory-reservation 256m \\
    \\
    # CPU: max 1.5 cores, higher priority than background workers
    --cpus 1.5 \\
    --cpu-shares 2048 \\
    \\
    # Process limit: prevent runaway forking
    --pids-limit 200 \\
    \\
    # Restart policy
    --restart unless-stopped \\
    \\
    myapp:prod
'''
print(production_run)

In [ ]:
# Actually run it and verify all limits are applied
!docker run -d \
    --name resource-check \
    --memory 256m \
    --memory-swap 256m \
    --cpus 0.75 \
    --cpu-shares 512 \
    --pids-limit 50 \
    nginx:alpine

import subprocess, json
data = json.loads(subprocess.check_output(["docker", "inspect", "resource-check"]))[0]
hc = data["HostConfig"]

print(f"Memory limit:      {hc['Memory'] // 1024**2} MB")
print(f"Memory+swap limit: {hc['MemorySwap'] // 1024**2} MB")
print(f"CPU quota:         {hc['NanoCpus'] / 1e9} cores")
print(f"CPU shares:        {hc['CpuShares']}")
print(f"PID limit:         {hc['PidsLimit']}")

!docker rm -f resource-check

## 6. Updating Limits on a Running Container

You can adjust most resource limits without restarting the container using `docker update`:

In [ ]:
# Start with conservative limits
!docker run -d --name dynamic-limits -m 64m --cpus 0.25 nginx:alpine

!docker inspect dynamic-limits --format \
    'Before — Memory: {{.HostConfig.Memory}} bytes  CPUs: {{.HostConfig.NanoCpus}}'

# Scale up limits in response to load (no restart needed)
!docker update --memory 256m --memory-swap 256m --cpus 1.0 dynamic-limits

!docker inspect dynamic-limits --format \
    'After  — Memory: {{.HostConfig.Memory}} bytes  CPUs: {{.HostConfig.NanoCpus}}'

!docker rm -f dynamic-limits

## 7. Reading Limits From Inside the Container

Some runtimes (JVM, Node.js) auto-tune based on available memory — but they read `/proc/meminfo` which shows host memory, not the container's limit. This causes the JVM to allocate a heap larger than the container allows, triggering OOM kills.

The fix: read from cgroup, not `/proc`.

In [ ]:
!docker run --rm -m 128m alpine sh -c '
    echo "=== /proc/meminfo (host memory — misleading inside containers) ==="
    grep MemTotal /proc/meminfo

    echo ""
    echo "=== cgroup limit (actual container limit) ==="
    if [ -f /sys/fs/cgroup/memory.max ]; then
        limit=$(cat /sys/fs/cgroup/memory.max)
        echo "cgroup v2: memory.max = $limit bytes ($(expr $limit / 1048576) MB)"
    else
        limit=$(cat /sys/fs/cgroup/memory/memory.limit_in_bytes)
        echo "cgroup v1: limit_in_bytes = $limit bytes ($(expr $limit / 1048576) MB)"
    fi
'

### Java and the JVM

Since Java 10, the JVM is cgroup-aware when run with `-XX:+UseContainerSupport` (enabled by default in Java 11+). It reads the cgroup limit and sizes the heap accordingly. Always use Java 11+ in containers.

```dockerfile
FROM eclipse-temurin:21-jre-alpine
# JVM reads cgroup memory limit automatically
# -XX:MaxRAMPercentage sets heap as % of the cgroup limit
CMD ["java", "-XX:MaxRAMPercentage=75.0", "-jar", "app.jar"]
```

With `-XX:MaxRAMPercentage=75.0` and a 512m container limit, the JVM allocates ~384 MB heap — leaving room for non-heap memory (metaspace, native threads, code cache).

## 8. Choosing Limits in Practice

There's no universal formula — the right limits depend on your workload. A practical approach:

### Step 1: Profile without limits

```bash
# Run under realistic load, observe actual usage
docker stats --no-stream myapp
```

### Step 2: Set limits with headroom

If peak memory usage is 200 MB, set `--memory 300m` — leave 50% headroom for spikes.

For CPU, set `--cpus` to the P95 usage plus some buffer. Don't set it to the peak — that causes throttling under normal load.

### Step 3: Monitor OOM kills in production

```bash
# Check if any running container has been OOM killed
docker ps -q | xargs docker inspect --format '{{.Name}} OOMKilled={{.State.OOMKilled}}' | grep true
```

An OOM kill in production means either your limit is too low or there is a memory leak. Investigate before just raising the limit.

### Rule of thumb per service type

| Service type | Memory guidance | CPU guidance |
|-------------|-----------------|-------------|
| Stateless API (Python/Node) | 1.5× peak observed | `--cpus` at P95 + 25% |
| JVM service | Set limit; use `-XX:MaxRAMPercentage=75` | `--cpus` at P95 + 50% |
| Nginx / proxy | 64–256 MB is usually ample | `--cpus 0.5–1` |
| Redis | Set to max allowed dataset + overhead | Low CPU needed |
| Batch / ML job | Allow spike headroom | `--cpus` equal to job parallelism |

## Summary

- **Memory:** `--memory` sets the hard RAM limit. When exceeded, the OOM killer terminates the process and Docker sets `OOMKilled: true`. `--memory-swap` controls RAM+swap total — set equal to `--memory` to disable swap.
- **CPU:** `--cpus` is a hard ceiling (e.g. `--cpus 1.5` = max 1.5 cores). `--cpu-shares` is a relative soft weight under contention (default 1024). `--cpuset-cpus` pins to specific cores.
- **Block I/O:** `--device-read-bps` / `--device-write-bps` throttle throughput. `--device-read-iops` / `--device-write-iops` throttle operations per second. `--blkio-weight` sets relative priority.
- **`--pids-limit`** caps the number of processes — prevents fork bombs.
- **`docker update`** adjusts memory, CPU, and restart policy on a running container without restart.
- **Cgroup vs `/proc/meminfo`:** inside a container, `/proc/meminfo` shows host memory — read from `/sys/fs/cgroup/memory.max` (v2) or `/sys/fs/cgroup/memory/memory.limit_in_bytes` (v1) for the actual container limit. JVM 11+ handles this automatically.
- **Sizing approach:** profile under realistic load with `docker stats`, set limits at 1.5× peak, monitor for OOM kills.